### P2.2 — ML Foundations | Python to Gen AI Course
### P2.2.4 – Supervised Learning: Classification (Naive Bayes Algorithm)

---
## 🔁 Quick Recap — Where We Are

In **P2.2.2**, we established how any ML problem breaks down:

<img src="ml.png" width="600"/>

**In this chapter, we cover one of the  Supervised Learning algorithms:**
- **Naive Bayes** — Classification task (detecting email spam)

> We won't build these algorithms from scratch — we use ready-made implementations  
> and focus on **when to use them**, **how they work**, and **how to evaluate them**.

---
## 🤖 What is Naive Bayes?

**Naive Bayes is a classification algorithm based on probability.**

The core idea: given some input features, what is the **probability** that this input belongs to each class? Whichever class has the higher probability wins.

It uses **Bayes' Theorem** under the hood:

```
                    P(Features | Class) × P(Class)
P(Class | Features) = ─────────────────────────────
                              P(Features)
```

**Why "Naive"?**  
It assumes each word is **independent** of the others

<img src="nb.png" width="700"/>

---
## 🌍 Real Life — Where Classification is Used Every Day

**📧 Gmail Spam Filter**
Gmail receives billions of emails daily. Nobody writes rules like:
```
IF email contains "Win money" THEN spam
```
There are infinite spam variations. Instead, Gmail trained a classifier on millions of labeled emails — spam vs not spam — and it now catches new spam it has never seen before.

**Other places classification runs silently:**
- 🏦 Your bank flags a transaction as fraud in milliseconds
- 🎵 Spotify classifies songs by mood (happy, sad, energetic)
- 🏥 AI screens X-rays as "normal" or "needs review"
- 💬 Social media detects hate speech automatically

> **Every time a computer puts something into a bucket — that's classification.**

---
## 🤖 Regression vs Classification — What's the Difference?

| | Regression | Classification |
|---|---|---|
| **Predicts** | A number | A category |
| **Example output** | ₹3,20,000 | "Spam" or "Ham" |
| **Algorithm used** | Linear Regression | Naive Bayes |
| **Evaluation** | RMSE, R2 Score | Accuracy, Confusion Matrix |

---

## 🔄 New Concept: Vectorization

ML models only understand **numbers**.  
But emails are **text**. So before training, we convert text → numbers.

```
"Win money now"        →  win, money, now
"Meeting at office"    →  meeting, at, office  
"Free prize offer"     →  free, prize, offer

Full vocabulary = [free, win, money, now, prize, at, meeting, offer]
                  8 words → 8-dimensional vector

                             word counts
"Win money now"           →  [0, 1, 1, 0, 1, 0, 0] 
"Meeting at office"       →  [0, 0, 0, 1, 0, 1, 1]
"Free prize offer"        →  [1, 0, 0, 0, 1, 0, 0]
```

This is called **CountVectorizer** — it counts how many times each word appears.  

In reality with thousands of emails the vector could be 50,000+ dimensions wide — one column per unique word ever seen.

```
CountVectorizer     →  raw word counts          ← what we used
TF-IDF              →  counts weighted by rarity
Word2Vec            →  words as geometric vectors
GloVe               →  meaning-aware embeddings
BERT Embeddings     →  context-aware deep vectors
```

These get progressively more powerful and complex — all of this will be covered in depth in NLP.

---

**So — which Classification algorithm do we use?**

We've confirmed: Supervised + Classification task. Now we need to pick the algorithm.

The standard approach is to **look at your data first** — understand what kind of input you have and what kind of boundary separates your classes:

For our spam detector: input is word counts from emails — Naive Bayes is purpose-built for this.

| What your data looks like | Algorithm to start with |
|---|---|
| Text / word frequencies | **Naive Bayes (MultinomialNB)** |
| Numbers with a clear linear boundary | Logistic Regression |
| Non-linear patterns, structured data | Decision Tree / Random Forest |
| High-dimensional, small dataset | SVM |

---

> **What if you can't analyse first?**  
> Same rule as Regression — **start with the simplest model** (Naive Bayes for text),  
> run the full ML workflow, check accuracy.  
> If accuracy is poor, try the next algorithm — same workflow, different algorithm.  
> You compare scores and pick the best. This is standard ML practice.

---
## 💻 Let's Build It — Spam Email Classifier

**Pipeline for this notebook (5 steps):**

<img src="spam.png" width="900"/>

We have 6 labeled emails:

```
Email                           Label
──────────────────────────────────────────
"Win money now"                 → Spam  🚫
"Limited offer win big"         → Spam  🚫
"Win a free prize now"          → Spam  🚫
"Meeting tomorrow at office"    → Ham   ✅
"Project discussion scheduled"  → Ham   ✅
"Let us plan the meeting"       → Ham   ✅
```

**Features (X):** The email text — vectorized into word counts  
**Labels (y):** `"spam"` or `"ham"` *(ham = not spam — legitimate email)*

> **Goal:** Train on labeled emails → predict whether a new unseen email is spam or ham.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── DATA ──────────────────────────────────────────────────────────
emails = [
    "Win money now",
    "Limited offer win big",
    "Win a free prize now",
    "Meeting tomorrow at office",
    "Project discussion scheduled",
    "Let us plan the meeting",
]
labels = ["Spam", "Spam", "Spam", "Ham", "Ham", "Ham"]


# ── VECTORIZATION (text → numbers) ───────────────────────────────
# Done before splitting — fits vocabulary on all emails
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(emails)

print("Vocabulary (words the model knows):")
print(vectorizer.get_feature_names_out())


# ── PHASE 1 : SPLIT ───────────────────────────────────────────────
# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)


# ── PHASE 2 : TRAIN ───────────────────────────────────────────────
# Model learns spam vs ham pattern from training data only
model = MultinomialNB()
model.fit(X_train, y_train)


# ── PHASE 3 : EVALUATE ────────────────────────────────────────────
# Pass both train and test data through the model
train_preds = model.predict(X_train)  #x_tran --> num ---> test ---> pred (test) spam , ham 
test_preds  = model.predict(X_test)


# Compute scores on both
train_accuracy = accuracy_score(y_train, train_preds)
test_accuracy  = accuracy_score(y_test,  test_preds)

print("\n=== EVALUATION ===")
print(f"Train Accuracy : {train_accuracy:.0%}")
print(f"Test  Accuracy : {test_accuracy:.0%}")

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, test_preds))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_preds))

# Compare scores — check for overfit / underfit
print("=== FIT CHECK ===")
if train_accuracy - test_accuracy > 0.15:
    print("⚠️  Overfitting  — model memorized training emails, may fail on new ones")
elif test_accuracy < 0.5:
    print("⚠️  Underfitting — model too simple, missed the spam pattern")
else:
    print("✅  Good Fit     — train and test scores are close, safe to proceed")


# ── PHASE 4 : INFERENCE ───────────────────────────────────────────
# Only run inference after confirming a good fit above
new_email = ["Win free money now"]
predicted = model.predict(vectorizer.transform(new_email))

print("\n=== INFERENCE ===")
print(f"Email      : '{new_email[0]}'")
print(f"Prediction : {predicted[0]}")

Vocabulary (words the model knows):
['at' 'big' 'discussion' 'free' 'let' 'limited' 'meeting' 'money' 'now'
 'offer' 'office' 'plan' 'prize' 'project' 'scheduled' 'the' 'tomorrow'
 'us' 'win']

=== EVALUATION ===
Train Accuracy : 100%
Test  Accuracy : 100%

Confusion Matrix (Test):
[[2]]

Classification Report (Test):
              precision    recall  f1-score   support

        Spam       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

=== FIT CHECK ===
✅  Good Fit     — train and test scores are close, safe to proceed

=== INFERENCE ===
Email      : 'Win free money now'
Prediction : Spam


D:\workspace\pycore\Course-GenAIFoundation\P2_LegacyToGenAI\P2.2_ML Foundations\.env_ml\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


---
## 🔍 Understanding the Results

**Accuracy**
- % of emails the model classified correctly
- `100%` on a 6-email dataset is expected — real-world accuracy is more meaningful with thousands of emails

**Confusion Matrix**
```
              Predicted Ham   Predicted Spam
Actual Ham  [      TN        |      FP      ]
Actual Spam [      FN        |      TP      ]
```
- **TP** (True Positive) — correctly caught spam ✅
- **TN** (True Negative) — correctly passed ham ✅
- **FP** (False Positive) — ham marked as spam ❌ (important email in junk!)
- **FN** (False Negative) — spam got through ❌

**Classification Report**
- **Precision** — of all emails flagged spam, how many were actually spam?
- **Recall** — of all actual spam, how many did we catch?
- **F1 Score** — balance between precision and recall (useful when dataset is imbalanced)

---

### 🎯 Is Your Model Fitting Correctly?

After every model, compare **Train Accuracy** vs **Test Accuracy**:

```
  Train ≈ Test (both high)   →  ✅ Good Fit     — learned the real pattern
  Train >> Test               →  ❌ Overfitting  — memorized training emails
  Train ≈ Test (both low)    →  ❌ Underfitting — model too simple
```

| Scenario | Train Accuracy | Test Accuracy | What it means |
|---|---|---|---|
| ✅ Good Fit | 95% | 92% | Generalizes to new emails |
| ❌ Overfitting | 100% | 60% | Memorized 4 training emails exactly |
| ❌ Underfitting | 55% | 52% | Couldn't find any spam pattern |

> With only 6 emails, 100% train accuracy is almost guaranteed — too few examples.  
> Real spam filters like Gmail train on **millions of emails**. Same concept, different scale.

---
## ✅ Pros & Limitations of Naive Bayes Classification

**Pros:**
- Extremely fast — works well even on large email datasets
- Handles text naturally — built for word-frequency style data
- Surprisingly effective despite its simplicity (used in real spam filters)

**Limitations:**
- Assumes words are independent of each other (not always true — "not good" vs "good")
- Struggles with small datasets — 6 emails is too few for reliable real-world accuracy
- Doesn't understand context or word meaning — only counts words